
# Benchmark: concurrent production-kernel groups for the six one-arm links

This notebook tests whether the six independent one-arm shifts can use more CPU
capacity by launching **multiple unchanged production kernels concurrently**.

It deliberately does not:

- change the WDM shift algebra;
- alter precision, lag truncation, or interpolation;
- introduce a link dimension into the large row-lag-frequency tensors;
- use signal localization or sparse masks.

The schedules are:

- one production call containing all six links;
- two concurrent production calls containing three links each;
- three concurrent production calls containing two links each.

The previously accepted two-thread plus/cross DWT route is also retained for
the final combined waveform-stage measurement.


In [1]:

# --- 1. Imports, CPU backend, and source checks ---

import inspect
import json
import os
import platform
import sys
import time
from datetime import datetime
from pathlib import Path

# This must be set before importing JAX, FEW, WDM, or WDM_LISAresponse.
os.environ["JAX_PLATFORMS"] = "cpu"

import jax
import jax.numpy as jnp
import numpy as np

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import psutil
except ImportError:
    psutil = None

from few.utils.constants import YRSID_SI
from few.waveform import GenerateEMRIWaveform
from lisaorbits.file import read_orbit_file
from lisaorbits import InterpolatedOrbits

from WDM_LISAresponse.code.lisa_response.waveforms import (
    generate_windowed_padded_waveform,
)
from WDM_LISAresponse.code.lisa_response.response_with_plan import (
    _build_wdm_dT_grid,
    _build_wdm_dT_context_from_wdm,
)
from WDM_LISAresponse.code.lisa_response.one_arm_plan import OneArmWDMPlan

import WDM
import WDM_LISAresponse
import WDM_LISAresponse.code.lisa_response.one_arm_plan as one_arm_plan_module
import WDM.code.time_delay_filters.plans as shift_plans_module


_process = psutil.Process(os.getpid()) if psutil is not None else None


def rss_gib():
    if _process is None:
        return np.nan
    return _process.memory_info().rss / 1024**3


print("Timestamp:", datetime.now().isoformat(timespec="seconds"))
print("Python executable:", sys.executable)
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("JAX:", jax.__version__)
print("JAX backend:", jax.default_backend())
print("JAX devices:", jax.devices())
print("WDM:", WDM.__file__)
print("WDM_LISAresponse:", WDM_LISAresponse.__file__)
print("plans.py:", shift_plans_module.__file__)
print("one_arm_plan.py:", one_arm_plan_module.__file__)
print("RSS:", f"{rss_gib():.2f} GiB")

assert jax.default_backend() == "cpu"
assert all(device.platform == "cpu" for device in jax.devices())
assert hasattr(
    shift_plans_module.VariableShiftBatchPlan,
    "apply_device_parallel_groups",
)
assert hasattr(
    OneArmWDMPlan,
    "apply_batch_device_materialized_parallel_groups",
)
assert "parallel_wdm_transforms" in inspect.signature(
    _build_wdm_dT_context_from_wdm
).parameters


Timestamp: 2026-07-17T14:21:16
Python executable: /data/tk593/conda-envs/lisatools/bin/python
Python: 3.12.12
Platform: Linux-5.14.0-687.24.1.el9_8.x86_64-x86_64-with-glibc2.34
JAX: 0.9.2
JAX backend: cpu
JAX devices: [CpuDevice(id=0)]
WDM: /data/tk593/LISA_Response/LISAresponse/WDM_GW_wavelets/src/WDM/__init__.py
WDM_LISAresponse: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/src/WDM_LISAresponse/__init__.py
plans.py: /data/tk593/LISA_Response/LISAresponse/WDM_GW_wavelets/src/WDM/code/time_delay_filters/plans.py
one_arm_plan.py: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/src/WDM_LISAresponse/code/lisa_response/one_arm_plan.py
RSS: 0.79 GiB


In [2]:

# --- 2. Full-year configuration and benchmark controls ---

N = 3_200_000
dt = 10
Nf = 100
q_wdm = 4
d_wdm = 4
A_frac = 0.25
L_TRUNC = 25

alpha = 0.005
tukey_factor = 4

ORBIT_FILE = "esa-trailing-orbits.h5"
theta = np.pi / 2
phi = 0.0

m1 = 1e6
m2 = 10.0
a = 0.9
p0 = 8.5
e0 = 0.3
x0 = 1.0
dist = 1.0
qK = np.pi / 3
phiK = np.pi / 3
Phi_phi0 = np.pi / 3
Phi_theta0 = 0.0
Phi_r0 = np.pi / 3
eps = 1e-2

BASE_SHIFT_KWARGS = {
    "assembly_backend": "lagfirst_chunked_lagblock",
    "assembly_precision": "complex64",
    "tl_tp_mode": "interp",
    "tl_tp_interp_points": 32,
    "tl_tp_interp_kind": "linear",
    "use_jax": True,
    "row_chunk_size": 32640,
    "lag_block_size": 17,
    "job_block_size": 1,
}

GROUP_CANDIDATES = [
    {
        "label": "one_group_of_six",
        "group_size": 6,
        "max_workers": 1,
    },
    {
        "label": "two_groups_of_three",
        "group_size": 3,
        "max_workers": 2,
    },
    {
        "label": "three_groups_of_two",
        "group_size": 2,
        "max_workers": 3,
    },
    {
        "label": "six_groups_of_one",
        "group_size": 1,
        "max_workers": 6,
    },
]

DWT_WARMUPS = 2
DWT_REPEATS = 15
SHIFT_WARMUPS = 2
SHIFT_REPEATS = 15
PIPELINE_WARMUPS = 2
PIPELINE_REPEATS = 11

MIN_MEDIAN_IMPROVEMENT_PERCENT = 3.0
MAX_REL_L2 = 1e-6

SAVE_RESULTS = False
RESULT_DIR = Path(
    ".benchmark_cache/one_arm_parallel_groups"
)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Expected WDM Nt:", N // (2 * Nf))
print("Number of lags:", 2 * L_TRUNC + 1)
print("Candidates:", GROUP_CANDIDATES)
print("Results:", RESULT_DIR)


Expected WDM Nt: 16000
Number of lags: 51
Candidates: [{'label': 'one_group_of_six', 'group_size': 6, 'max_workers': 1}, {'label': 'two_groups_of_three', 'group_size': 3, 'max_workers': 2}, {'label': 'three_groups_of_two', 'group_size': 2, 'max_workers': 3}, {'label': 'six_groups_of_one', 'group_size': 1, 'max_workers': 6}]
Results: .benchmark_cache/one_arm_parallel_groups


In [3]:

# --- 3. Timing, CPU-use, validation, and display helpers ---

def block_result(value):
    for leaf in jax.tree_util.tree_leaves(value):
        if hasattr(leaf, "block_until_ready"):
            leaf.block_until_ready()
    return value


def process_cpu_seconds():
    if _process is None:
        return np.nan
    values = _process.cpu_times()
    return float(values.user + values.system)


def timed_call(function):
    cpu_start = process_cpu_seconds()
    wall_start = time.perf_counter()

    result = function()
    block_result(result)

    wall_s = time.perf_counter() - wall_start
    cpu_end = process_cpu_seconds()
    cpu_s = (
        float(cpu_end - cpu_start)
        if np.isfinite(cpu_start) and np.isfinite(cpu_end)
        else np.nan
    )
    effective_cores = (
        cpu_s / wall_s
        if wall_s > 0 and np.isfinite(cpu_s)
        else np.nan
    )
    return result, float(wall_s), float(cpu_s), float(effective_cores)


def summarize(values):
    values = np.asarray(values, dtype=float)
    q25, q75 = np.percentile(values, [25.0, 75.0])
    return {
        "median": float(np.median(values)),
        "mean": float(np.mean(values)),
        "minimum": float(np.min(values)),
        "maximum": float(np.max(values)),
        "q25": float(q25),
        "q75": float(q75),
        "iqr": float(q75 - q25),
        "all": [float(value) for value in values],
    }


def compare_device_batches(candidate, reference):
    candidate = jnp.asarray(candidate)
    reference = jnp.asarray(reference)

    delta = candidate - reference
    max_abs = jnp.max(jnp.abs(delta))
    reference_norm = jnp.linalg.norm(reference)
    rel_l2 = jnp.where(
        reference_norm > 0,
        jnp.linalg.norm(delta) / reference_norm,
        jnp.nan,
    )
    max_abs.block_until_ready()
    rel_l2.block_until_ready()

    return {
        "max_abs": float(max_abs),
        "rel_l2": float(rel_l2),
    }


def assess_candidate(baseline_wall, candidate_wall, *, rel_l2):
    baseline = summarize(baseline_wall)
    candidate = summarize(candidate_wall)

    runtime_change_percent = 100.0 * (
        candidate["median"] / baseline["median"] - 1.0
    )
    improvement_percent = -runtime_change_percent
    numerically_valid = bool(
        np.isfinite(rel_l2) and rel_l2 <= MAX_REL_L2
    )
    iqr_separated = bool(candidate["q75"] < baseline["q25"])

    if (
        numerically_valid
        and improvement_percent >= MIN_MEDIAN_IMPROVEMENT_PERCENT
        and iqr_separated
    ):
        decision = "adopt"
    elif (
        numerically_valid
        and improvement_percent >= MIN_MEDIAN_IMPROVEMENT_PERCENT
    ):
        decision = "retest"
    else:
        decision = "retain_baseline"

    return {
        "baseline": baseline,
        "candidate": candidate,
        "runtime_change_percent": float(runtime_change_percent),
        "improvement_percent": float(improvement_percent),
        "numerically_valid": numerically_valid,
        "iqr_separated": iqr_separated,
        "decision": decision,
    }


def paired_benchmark(
    baseline_function,
    candidate_function,
    *,
    warmups,
    repeats,
):
    for _ in range(int(warmups)):
        block_result(baseline_function())
        block_result(candidate_function())

    baseline_wall = []
    baseline_cpu = []
    baseline_effective = []
    candidate_wall = []
    candidate_cpu = []
    candidate_effective = []

    baseline_output = None
    candidate_output = None

    for repeat_index in range(int(repeats)):
        routes = (
            (
                ("baseline", baseline_function),
                ("candidate", candidate_function),
            )
            if repeat_index % 2 == 0
            else (
                ("candidate", candidate_function),
                ("baseline", baseline_function),
            )
        )

        for label, function in routes:
            output, wall_s, cpu_s, effective_cores = timed_call(function)

            if label == "baseline":
                baseline_output = output
                baseline_wall.append(wall_s)
                baseline_cpu.append(cpu_s)
                baseline_effective.append(effective_cores)
            else:
                candidate_output = output
                candidate_wall.append(wall_s)
                candidate_cpu.append(cpu_s)
                candidate_effective.append(effective_cores)

    return {
        "baseline_output": baseline_output,
        "candidate_output": candidate_output,
        "baseline_wall": baseline_wall,
        "baseline_cpu": baseline_cpu,
        "baseline_effective": baseline_effective,
        "candidate_wall": candidate_wall,
        "candidate_cpu": candidate_cpu,
        "candidate_effective": candidate_effective,
    }


def show_rows(rows):
    if pd is not None:
        frame = pd.DataFrame(rows)
        display(frame)
        return frame
    for row in rows:
        print(row)
    return rows


## 4. Generate one fixed full-year waveform and orbit object

In [ ]:

# --- 4a. Orbit and waveform generator ---

td_gen = GenerateEMRIWaveform(
    "FastKerrEccentricEquatorialFlux",
    sum_kwargs=dict(pad_output=True, odd_len=True),
    return_list=True,
)

orbit_data = read_orbit_file(ORBIT_FILE)
orbit_times = orbit_data.t0 + orbit_data.dt * np.arange(orbit_data.size)
orbits_trailing = InterpolatedOrbits(
    orbit_times,
    orbit_data.positions,
    orbit_data.velocities,
)

t0 = orbit_data.t0 + 1.0 * YRSID_SI

model_kwargs = {
    "td_gen": td_gen,
    "m1": m1,
    "m2": m2,
    "a": a,
    "p0": p0,
    "e0": e0,
    "x0": x0,
    "dist": dist,
    "theta": theta,
    "phi": phi,
    "qK": qK,
    "phiK": phiK,
    "Phi_phi0": Phi_phi0,
    "Phi_theta0": Phi_theta0,
    "Phi_r0": Phi_r0,
    "eps": eps,
}

window_kwargs = {
    "alpha": alpha,
    "tukey_factor": tukey_factor,
}


In [ ]:

# --- 4b. Generate and pad the waveform ---

started = time.perf_counter()

times_pad, hplus_td, hcross_td, pad_info = (
    generate_windowed_padded_waveform(
        waveform_model="emri",
        t0=t0,
        dt=dt,
        N=N,
        model_kwargs=model_kwargs,
        window_function="tukey",
        window_kwargs=window_kwargs,
        wdm_Nf=Nf,
        return_padding_info=True,
    )
)

waveform_generation_s = time.perf_counter() - started

print("Waveform generation_s:", f"{waveform_generation_s:.3f}")
print("times_pad:", times_pad.shape, times_pad.dtype)
print("hplus_td:", hplus_td.shape, hplus_td.dtype)
print("hcross_td:", hcross_td.shape, hcross_td.dtype)
print("WDM Nt x Nf:", pad_info["wdm_Nt"], "x", Nf)
print("RSS:", f"{rss_gib():.2f} GiB")


## 5. Build the WDM grid and confirm parallel plus/cross DWTs

In [ ]:

# --- 5. WDM grid and DWT comparison ---

started = time.perf_counter()
grid_context = _build_wdm_dT_grid(
    times_pad,
    Nf=Nf,
    d=d_wdm,
    A_frac=A_frac,
    q=q_wdm,
    calc_m0=True,
)
grid_build_s = time.perf_counter() - started
wdm = grid_context["wdm"]


def run_sequential_context():
    return _build_wdm_dT_context_from_wdm(
        times_pad,
        hplus_td,
        hcross_td,
        wdm,
        parallel_wdm_transforms=False,
    )


def run_parallel_context():
    return _build_wdm_dT_context_from_wdm(
        times_pad,
        hplus_td,
        hcross_td,
        wdm,
        parallel_wdm_transforms=True,
    )


dwt_paired = paired_benchmark(
    run_sequential_context,
    run_parallel_context,
    warmups=DWT_WARMUPS,
    repeats=DWT_REPEATS,
)

sequential_context = dwt_paired["baseline_output"]
parallel_context = dwt_paired["candidate_output"]

plus_comparison = compare_device_batches(
    jnp.asarray(parallel_context["w_t_plus"])[None, ...],
    jnp.asarray(sequential_context["w_t_plus"])[None, ...],
)
cross_comparison = compare_device_batches(
    jnp.asarray(parallel_context["w_t_cross"])[None, ...],
    jnp.asarray(sequential_context["w_t_cross"])[None, ...],
)

dwt_assessment = assess_candidate(
    dwt_paired["baseline_wall"],
    dwt_paired["candidate_wall"],
    rel_l2=max(
        plus_comparison["rel_l2"],
        cross_comparison["rel_l2"],
    ),
)

print("WDM grid build_s:", f"{grid_build_s:.3f}")
print(
    "Sequential DWT median_s:",
    f"{summarize(dwt_paired['baseline_wall'])['median']:.3f}",
)
print(
    "Parallel DWT median_s:",
    f"{summarize(dwt_paired['candidate_wall'])['median']:.3f}",
)
print(
    "Runtime change:",
    f"{dwt_assessment['runtime_change_percent']:+.2f}%",
)
print("Plus comparison:", plus_comparison)
print("Cross comparison:", cross_comparison)
print("Decision:", dwt_assessment["decision"])

w_plus_device = jnp.asarray(
    parallel_context["w_t_plus"],
    dtype=jnp.complex64,
)
w_cross_device = jnp.asarray(
    parallel_context["w_t_cross"],
    dtype=jnp.complex64,
)
block_result((w_plus_device, w_cross_device))


## 6. Build the unchanged production one-arm plan

In [ ]:

# --- 6. Production one-arm plan ---

started = time.perf_counter()
one_arm_plan = OneArmWDMPlan.build_from_context(
    grid_context,
    orbits_trailing,
    theta,
    phi,
    L_trunc=L_TRUNC,
    variable_shift_kwargs=BASE_SHIFT_KWARGS,
    use_jax=True,
)
plan_build_s = time.perf_counter() - started


def run_production_shift():
    return one_arm_plan.apply_batch_device_materialized(
        w_plus_device,
        w_cross_device,
    )


reference_device, baseline_wall, baseline_cpu, baseline_effective = (
    timed_call(run_production_shift)
)

# A second call excludes first-execution effects before candidate screening.
reference_device, baseline_wall, baseline_cpu, baseline_effective = (
    timed_call(run_production_shift)
)

print("Plan build_s:", f"{plan_build_s:.3f}")
print("Baseline shift_s:", f"{baseline_wall:.3f}")
print("Baseline effective cores:", f"{baseline_effective:.2f}")
print("Reference:", reference_device.shape, reference_device.dtype)
print(
    "Persistent plan memory_MiB:",
    one_arm_plan.shift_plan.device_plan_memory_bytes / 1024**2,
)



## 7. Screen concurrent production-kernel groups

The candidate calls the same production dispatcher on disjoint contiguous link
groups. Each worker synchronizes its own JAX result, so the measured wall time
includes actual concurrent CPU execution rather than asynchronous enqueueing.


In [ ]:

# --- 7. Paired shift benchmark ---

shift_rows = []
shift_details = {}

for candidate in GROUP_CANDIDATES:
    label = candidate["label"]
    group_size = int(candidate["group_size"])
    max_workers = int(candidate["max_workers"])
    num_groups = (
        one_arm_plan.shift_plan.num_jobs + group_size - 1
    ) // group_size

    def run_grouped_shift(
        group_size=group_size,
        max_workers=max_workers,
    ):
        return (
            one_arm_plan
            .apply_batch_device_materialized_parallel_groups(
                w_plus_device,
                w_cross_device,
                group_size=group_size,
                max_workers=max_workers,
            )
        )

    paired = paired_benchmark(
        run_production_shift,
        run_grouped_shift,
        warmups=SHIFT_WARMUPS,
        repeats=SHIFT_REPEATS,
    )

    comparison = compare_device_batches(
        paired["candidate_output"],
        paired["baseline_output"],
    )
    assessment = assess_candidate(
        paired["baseline_wall"],
        paired["candidate_wall"],
        rel_l2=comparison["rel_l2"],
    )

    baseline_effective_summary = summarize(
        paired["baseline_effective"]
    )
    candidate_effective_summary = summarize(
        paired["candidate_effective"]
    )

    row = {
        "candidate": label,
        "num_groups": int(num_groups),
        "links_per_group": group_size,
        "max_workers": max_workers,
        "baseline_median_s": assessment["baseline"]["median"],
        "candidate_median_s": assessment["candidate"]["median"],
        "runtime_change_percent": assessment[
            "runtime_change_percent"
        ],
        "speedup": (
            assessment["baseline"]["median"]
            / assessment["candidate"]["median"]
        ),
        "baseline_effective_cores_median": (
            baseline_effective_summary["median"]
        ),
        "candidate_effective_cores_median": (
            candidate_effective_summary["median"]
        ),
        "effective_core_change": (
            candidate_effective_summary["median"]
            - baseline_effective_summary["median"]
        ),
        "baseline_q25_s": assessment["baseline"]["q25"],
        "baseline_q75_s": assessment["baseline"]["q75"],
        "candidate_q25_s": assessment["candidate"]["q25"],
        "candidate_q75_s": assessment["candidate"]["q75"],
        "iqr_separated": assessment["iqr_separated"],
        "max_abs": comparison["max_abs"],
        "rel_l2": comparison["rel_l2"],
        "decision": assessment["decision"],
    }
    shift_rows.append(row)
    shift_details[label] = {
        "paired": paired,
        "comparison": comparison,
        "assessment": assessment,
    }

    print(f"\nCandidate: {label}")
    print(
        "  schedule:",
        f"{num_groups} group(s) × {group_size} link(s)",
    )
    print(
        "  baseline median:",
        f"{row['baseline_median_s']:.3f} s",
    )
    print(
        "  candidate median:",
        f"{row['candidate_median_s']:.3f} s",
    )
    print(
        "  runtime change:",
        f"{row['runtime_change_percent']:+.2f}%",
    )
    print(
        "  effective cores:",
        f"{row['baseline_effective_cores_median']:.2f}"
        " -> "
        f"{row['candidate_effective_cores_median']:.2f}",
    )
    print("  rel_l2:", f"{row['rel_l2']:.3e}")
    print("  decision:", row["decision"])

shift_table = show_rows(shift_rows)



## 8. Combined waveform-dependent stage

The accepted baseline for this section is:

```text
parallel plus/cross DWT + unchanged production six-link shift
```

A grouped-shift pipeline is measured only when a grouped schedule passes the
runtime and numerical adoption guard. A rejected candidate is never
automatically substituted into the combined route.


In [ ]:

# --- 8. Combined DWT + one-arm comparison ---

adopted_group_rows = [
    row for row in shift_rows
    if row["decision"] == "adopt"
]

best_adopted_group = (
    min(adopted_group_rows, key=lambda row: row["candidate_median_s"])
    if adopted_group_rows
    else None
)


def run_old_pipeline():
    context = run_sequential_context()
    return one_arm_plan.apply_batch_device_materialized(
        context["w_t_plus"],
        context["w_t_cross"],
    )


def run_accepted_pipeline():
    context = run_parallel_context()
    return one_arm_plan.apply_batch_device_materialized(
        context["w_t_plus"],
        context["w_t_cross"],
    )


accepted_pipeline_pair = paired_benchmark(
    run_old_pipeline,
    run_accepted_pipeline,
    warmups=PIPELINE_WARMUPS,
    repeats=PIPELINE_REPEATS,
)
accepted_pipeline_comparison = compare_device_batches(
    accepted_pipeline_pair["candidate_output"],
    accepted_pipeline_pair["baseline_output"],
)
accepted_pipeline_assessment = assess_candidate(
    accepted_pipeline_pair["baseline_wall"],
    accepted_pipeline_pair["candidate_wall"],
    rel_l2=accepted_pipeline_comparison["rel_l2"],
)

pipeline_rows = [{
    "route": "parallel_dwt_plus_production_shift",
    "baseline_median_s": accepted_pipeline_assessment[
        "baseline"
    ]["median"],
    "candidate_median_s": accepted_pipeline_assessment[
        "candidate"
    ]["median"],
    "runtime_change_percent": accepted_pipeline_assessment[
        "runtime_change_percent"
    ],
    "rel_l2": accepted_pipeline_comparison["rel_l2"],
    "decision": accepted_pipeline_assessment["decision"],
}]

if best_adopted_group is not None:
    best_group_size = int(best_adopted_group["links_per_group"])
    best_max_workers = int(best_adopted_group["max_workers"])

    def run_grouped_pipeline():
        context = run_parallel_context()
        return (
            one_arm_plan
            .apply_batch_device_materialized_parallel_groups(
                context["w_t_plus"],
                context["w_t_cross"],
                group_size=best_group_size,
                max_workers=best_max_workers,
            )
        )

    grouped_pipeline_pair = paired_benchmark(
        run_accepted_pipeline,
        run_grouped_pipeline,
        warmups=PIPELINE_WARMUPS,
        repeats=PIPELINE_REPEATS,
    )
    grouped_pipeline_comparison = compare_device_batches(
        grouped_pipeline_pair["candidate_output"],
        grouped_pipeline_pair["baseline_output"],
    )
    grouped_pipeline_assessment = assess_candidate(
        grouped_pipeline_pair["baseline_wall"],
        grouped_pipeline_pair["candidate_wall"],
        rel_l2=grouped_pipeline_comparison["rel_l2"],
    )

    pipeline_rows.append({
        "route": (
            "parallel_dwt_plus_grouped_shift_"
            + best_adopted_group["candidate"]
        ),
        "baseline_median_s": grouped_pipeline_assessment[
            "baseline"
        ]["median"],
        "candidate_median_s": grouped_pipeline_assessment[
            "candidate"
        ]["median"],
        "runtime_change_percent": grouped_pipeline_assessment[
            "runtime_change_percent"
        ],
        "rel_l2": grouped_pipeline_comparison["rel_l2"],
        "decision": grouped_pipeline_assessment["decision"],
    })
else:
    grouped_pipeline_pair = None
    grouped_pipeline_comparison = None
    grouped_pipeline_assessment = None
    print(
        "No concurrent-group schedule passed the adoption guard; "
        "the combined route remains parallel DWT + production shift."
    )

pipeline_table = show_rows(pipeline_rows)


## 9. Save a compact result record and interpret the decision

In [ ]:

# --- 9. Results and decision summary ---

results = {
    "configuration": {
        "N": N,
        "dt": dt,
        "Nf": Nf,
        "q_wdm": q_wdm,
        "d_wdm": d_wdm,
        "A_frac": A_frac,
        "L_TRUNC": L_TRUNC,
        "BASE_SHIFT_KWARGS": BASE_SHIFT_KWARGS,
        "GROUP_CANDIDATES": GROUP_CANDIDATES,
    },
    "environment": {
        "python": sys.version,
        "platform": platform.platform(),
        "jax": jax.__version__,
        "jax_backend": jax.default_backend(),
        "plans_file": shift_plans_module.__file__,
        "one_arm_plan_file": one_arm_plan_module.__file__,
    },
    "dwt": {
        "assessment": dwt_assessment,
        "plus_comparison": plus_comparison,
        "cross_comparison": cross_comparison,
    },
    "shift_rows": shift_rows,
    "pipeline_rows": pipeline_rows,
}

result_path = RESULT_DIR / "one_arm_parallel_groups_results.json"
if SAVE_RESULTS:
    result_path.write_text(
        json.dumps(results, indent=2),
        encoding="utf-8",
    )
    print("Saved:", result_path)

print("\n=== Decision summary ===")
print(
    "Parallel DWT:",
    dwt_assessment["decision"],
    f"({dwt_assessment['runtime_change_percent']:+.2f}%)",
)

if best_adopted_group is None:
    print(
        "Concurrent link groups: retain the single six-link production call."
    )
    print(
        "Interpretation: the apparent CPU headroom is not converted into "
        "lower latency by independent concurrent production calls. Shared "
        "CPU/cache/memory resources or JAX's CPU scheduler are the likely "
        "constraint."
    )
else:
    print(
        "Best concurrent schedule:",
        best_adopted_group["candidate"],
        f"({best_adopted_group['runtime_change_percent']:+.2f}%)",
    )
    print(
        "Interpretation: outer link parallelism is useful, while the previous "
        "rank-four link-block formulation was the inefficient implementation."
    )
